In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
df_gdp = pd.read_csv("../DATA/gdp_pcap.csv")
df_co2 = pd.read_csv("../DATA/co2_pcap_cons.csv")

# Drop the country abbreviation column
df_gdp = df_gdp.drop(columns=df_gdp.columns[0])
df_co2 = df_co2.drop(columns=df_co2.columns[0])

# Set country names as index
df_gdp = df_gdp.set_index(df_gdp.columns[0])
df_co2 = df_co2.set_index(df_co2.columns[0])

df_gdp.head()

,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,...,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100
name,,,,,,,,,,,,,,,,,,,,,
Afghanistan,560.88817,560.88817,560.88817,560.88817,560.88817,560.88817,560.88817,560.88817,560.88817,560.88817,...,9397.79550,9628.20479,9864.18612,10105.83025,10353.22669,10606.46351,10865.62717,11130.80231,11402.07162,11679.51560
Angola,435.23259,436.64111,438.75389,440.86667,442.27519,444.38797,446.50074,448.61352,450.72630,452.13482,...,30471.68022,30991.50308,31513.98698,32038.95027,32566.20896,33095.57701,33626.86659,34159.88835,34694.45172,35230.36514
Albania,547.53369,549.10393,550.67868,552.25795,553.84175,555.43009,557.02298,558.62044,560.22248,561.82912,...,57444.50359,57884.32605,58319.28840,58749.34444,59174.45223,59594.57402,60009.67615,60419.72899,60824.70684,61224.58783
Andorra,1598.53128,1601.20217,1603.87307,1607.87941,1610.55031,1613.22120,1615.89210,1618.56300,1622.56934,1625.24023,...,82535.68235,82627.86812,82718.08635,82806.37556,82892.77367,82977.31800,83060.04528,83140.99165,83220.19266,83297.68327
UAE,1332.77712,1336.78346,1342.12526,1347.46705,1352.80884,1356.81518,1362.15698,1367.49877,1372.84056,1378.18235,...,81900.09821,81885.61721,81871.47395,81857.66048,81844.16902,81830.99199,81818.12198,81805.55176,81793.27428,81781.28265


In [4]:
print(df_gdp.index.difference(df_co2.index))
print(df_co2.index.difference(df_gdp.index))

Index(['Monaco', 'San Marino'], dtype='object', name='name')
Index(['Liechtenstein'], dtype='object', name='name')


In [5]:
common = df_gdp.index.intersection(df_co2.index)

df_gdp = df_gdp.loc[common]
df_co2 = df_co2.loc[common]

print(df_gdp.index.difference(df_co2.index))
print(df_co2.index.difference(df_gdp.index))

Index([], dtype='object', name='name')
Index([], dtype='object', name='name')


In [6]:
common_years = df_gdp.columns.intersection(df_co2.columns)

df_gdp = df_gdp[common_years]
df_co2 = df_co2[common_years]

df_gdp.columns.equals(df_co2.columns)

common_years = common_years.astype(int)

In [62]:
def plot_country(country): 
    plt.figure(figsize=(7,5)) 
    
    scatter = plt.scatter(df_gdp.loc[country], df_co2.loc[country], s=30, c = common_years) 
    
    plt.xlabel("GDP per Capita (constant international $ (2021))", fontsize=12) 
    plt.ylabel("CO₂ Emissions per Capita (tonnes per capita)", fontsize=12) 
    plt.title(f"{country}: CO2 vs GDP per capita", fontsize=14) 
    plt.colorbar(scatter, label="Year") 
    
    plt.grid(alpha=0.3) 
    plt.tight_layout() 
    plt.show()

def plot_country_px(country):
    df = pd.DataFrame({"GDP": df_gdp.loc[country], "CO2": df_co2.loc[country], "Year": common_years
    })

    fig = px.line(df, x="GDP", y="CO2", markers=True, hover_data=["Year"], title=f"{country}: CO2 vs GDP per capita")
    
    fig.update_traces(
        marker=dict(
            color=df["Year"],
            colorscale="viridis",
            size=7,
            colorbar=dict(title="Year")
        )
    )
    
    fig.update_layout(width=800,height=600, hovermode="x unified", margin=dict(l=50, r=50, t=50, b=40))
    
    fig.show()

plot_country_px("USA")

In [36]:
def plot_gdp_co2(country):
    gdp = df_gdp.loc[country].astype(float)
    co2 = df_co2.loc[country].astype(float)
    years = gdp.index.astype(int)
    
    fig, ax1 = plt.subplots(figsize=(8,5))
    
    # Left axis: GDP
    ax1.set_xlabel("Year")
    ax1.set_ylabel("GDP per Capita (USD)", color="blue")
    ax1.plot(years, gdp, color="blue", label="GDP per Capita")
    ax1.tick_params(axis='y', labelcolor="blue")
    
    # Right axis: CO2
    ax2 = ax1.twinx()
    ax2.set_ylabel("CO₂ per Capita", color="green")
    ax2.plot(years, co2, color="green", label="CO₂ per Capita")
    ax2.tick_params(axis='y', labelcolor="green")
    
    # Title and grid
    plt.title(f"{country}: GDP and CO₂ per Capita Over Time", fontsize=14)
    ax1.grid(alpha=0.3)
    
    # Combine legends from both axes
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc="upper left")
    
    plt.tight_layout()
    plt.show()
    
def plot_gdp_co2_px(country):
    gdp = df_gdp.loc[country].astype(float)
    co2 = df_co2.loc[country].astype(float)
    years = gdp.index.astype(int)

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # GDP line
    fig.add_trace(go.Scatter(x=years, y=gdp, mode="lines", name="GDP per Capita", line=dict(color="blue"), 
                   hovertemplate="GDP: %{y}<extra></extra>"), secondary_y=False)

    # CO2 line
    fig.add_trace(go.Scatter(x=years, y=co2, mode="lines", name="CO₂ per Capita",line=dict(color="green"),
            hovertemplate="CO₂: %{y}<extra></extra>"), secondary_y=True)

    fig.update_layout(title=f"{country}: GDP and CO₂ per Capita Over Time", width=800,
        height=500, xaxis_title="Year", hovermode="x unified", margin=dict(l=50, r=50, t=50, b=40))

    fig.update_yaxes(title_text="GDP per Capita (USD)", secondary_y=False)

    fig.update_yaxes(title_text="CO₂ per Capita", secondary_y=True)

    fig.show()

In [37]:
plot_gdp_co2_px("USA")

In [44]:
plot_country_px("China")

In [45]:
plot_gdp_co2_px("China")

In [46]:
plot_country_px("Brazil")

In [47]:
plot_gdp_co2_px("Brazil")

In [49]:
plot_country_px("Germany")

In [50]:
plot_gdp_co2_px("Germany")

In [51]:
plot_country_px("India")

In [52]:
plot_gdp_co2_px("India")

In [53]:
plot_country_px("South Africa")

In [54]:
plot_gdp_co2_px("South Africa")